In [ ]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/hahn_2023/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘data.table’


The following objects are masked from ‘package:reshape2’:

    dcast, melt


The following objects are masked from ‘package:dplyr’:

    between, first, last




Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [ ]:
n_workers <- 20
mod_def <- "TopModPosBC"

max_set_size <- 500
min_set_size <- 3

## Prep marker genes

### Claude marker genes

In [ ]:
# set_source <- "Claude_marker_genes"

# marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_mouse.RDS")

### MO marker genes

In [ ]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

mask <- !MO_legend$Category %in% "Disease"
marker_genes_list <- MO_sets[mask]
names(marker_genes_list) <- MO_legend$SetName[mask]

set_sizes <- lengths(marker_genes_list)
mask <- (set_sizes < max_set_size) & (set_sizes > min_set_size)
marker_genes_list <- marker_genes_list[mask]

print(length(marker_genes_list))

set_source <- paste0("MO_", length(marker_genes_list), "sets")

[1] 1271


## Get enrichments

In [ ]:
network_dir <- "hahn_2023_cortex_STAR_counts_TMMF_All_130_outliers_removed_44764genes_cleaned_mergeParam0.93_minSignum0.88_Modules"

enrichments_df <- get_module_enrichments_par(network_dir, marker_genes_list, mod_def, n_workers)

write.csv(enrichments_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)
    
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments_top_Qval_mods.csv"), row.names=FALSE, quote=FALSE)